# 한 화면 고객 대시보드 (예제 03 보충)

카드 3장 · 히트맵 · 막대 4개를 **한 화면**에 그리고, 직업을 골라 다시 볼 수 있습니다.

| 순서 | 셀 | 하는 일 |
|:---:|---|---|
| 1 | 0번 셀 | 데이터를 내려받고 대시보드 함수를 만듭니다 (실행만) |
| 2 | 1번 셀 | 직업 선택 상자가 나옵니다 → 직업을 고릅니다 |
| 3 | 2번 셀 | 고른 직업으로 대시보드를 그립니다 |
| 4 | 3번 셀 | 고른 직업의 고객 표를 변수로 받아 더 살펴봅니다 |

> 직업을 바꾸면 **2번 셀부터 다시 실행**하세요. 셀은 실행하는 순간의 선택을 씁니다.

## 0. 실습 준비 — 데이터와 대시보드 함수

In [ ]:
# 0번 셀: 데이터 준비 + 대시보드 함수 만들기 (실행만 하세요)
import os
import urllib.request
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1) 데이터 준비 (없으면 GitHub에서 내려받기)
DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILE = "bank_customers_prepared.csv"
if not os.path.exists(FILE):
    try:
        urllib.request.urlretrieve(DATA_URL + FILE, FILE)
    except Exception:                 # 인터넷이 막혔을 때만 직접 올리기
        from google.colab import files
        files.upload()
customers = pd.read_csv(FILE)

# 한글판 CSV(직업·연령대·잔액구간)도, 영문판 CSV(job_name·age_group·balance_group)도 읽히게 이름을 맞춥니다
customers = customers.rename(columns={"직업": "job_name", "연령대": "age_group", "잔액구간": "balance_group"})

# 2) 구간 순서와 새 항목 만들기
AGE_ORDER = ["20대 이하", "30대", "40대", "50대", "60대 이상"]
BALANCE_ORDER = ["마이너스", "0유로", "1~1,000", "1,001~5,000", "5,000 초과"]
LOAN_ORDER = ["대출 없음", "주택담보만", "개인대출만", "둘 다"]
CAMPAIGN_NAMES = {"success": "이전 가입", "failure": "이전 미가입", "other": "기타", "unknown": "이전 연락 없음"}

customers["age_group"] = pd.Categorical(customers["age_group"], categories=AGE_ORDER, ordered=True)
customers["balance_group"] = pd.Categorical(customers["balance_group"], categories=BALANCE_ORDER, ordered=True)

customers["loan_status"] = "대출 없음"
customers.loc[(customers["housing"] == "yes") & (customers["loan"] == "no"), "loan_status"] = "주택담보만"
customers.loc[(customers["housing"] == "no") & (customers["loan"] == "yes"), "loan_status"] = "개인대출만"
customers.loc[(customers["housing"] == "yes") & (customers["loan"] == "yes"), "loan_status"] = "둘 다"
customers["loan_status"] = pd.Categorical(customers["loan_status"], categories=LOAN_ORDER, ordered=True)

customers["prev_campaign"] = pd.Categorical(customers["poutcome"].map(CAMPAIGN_NAMES),
                                            categories=list(CAMPAIGN_NAMES.values()), ordered=True)

# 막대 4개에 쓸 기준과 화면 제목
BAR_COLS = {"age_group": "연령대", "balance_group": "잔액구간", "loan_status": "대출상태", "prev_campaign": "과거캠페인"}


# 3) 가입률표: 고객수 · 가입자수 · 가입률(%)
def rate_table(data, cols):
    table = data.groupby(cols, observed=False)["y"].agg(
        n_customers="size",
        n_subscribed=lambda x: (x == "yes").sum(),
    )
    table["rate_pct"] = (table["n_subscribed"] / table["n_customers"] * 100).round(1)
    return table


# 4) 한 화면 대시보드
def full_dashboard(job="전체"):
    data = customers if job == "전체" else customers[customers["job_name"] == job]
    n_customers = len(data)
    n_subscribed = int((data["y"] == "yes").sum())
    overall_rate = n_subscribed / n_customers * 100

    # 6칸 격자: 카드는 2칸씩, 히트맵은 6칸 전체, 막대는 3칸씩
    fig = make_subplots(
        rows=4, cols=6,
        specs=[[{"type": "indicator", "colspan": 2}, None, {"type": "indicator", "colspan": 2}, None,
                {"type": "indicator", "colspan": 2}, None],
               [{"type": "heatmap", "colspan": 6}, None, None, None, None, None],
               [{"colspan": 3}, None, None, {"colspan": 3}, None, None],
               [{"colspan": 3}, None, None, {"colspan": 3}, None, None]],
        row_heights=[0.12, 0.32, 0.28, 0.28],
        vertical_spacing=0.07,
        subplot_titles=["", "", "", "연령대 × 잔액구간 가입률(%)"] + [f"{name}별 가입률(%)" for name in BAR_COLS.values()],
    )

    # ① 카드 3장
    fig.add_trace(go.Indicator(mode="number", value=n_customers, title={"text": "고객 수(명)"},
                               number={"valueformat": ","}), row=1, col=1)
    fig.add_trace(go.Indicator(mode="number", value=n_subscribed, title={"text": "가입자 수(명)"},
                               number={"valueformat": ","}), row=1, col=3)
    fig.add_trace(go.Indicator(mode="number", value=overall_rate, title={"text": "정기예금 가입률"},
                               number={"suffix": "%", "valueformat": ".1f"}), row=1, col=5)

    # ② 히트맵: 연령대 × 잔액구간
    grid = rate_table(data, ["age_group", "balance_group"])
    rate_grid = grid["rate_pct"].unstack()
    count_grid = grid["n_customers"].unstack()
    cell_text = rate_grid.map(lambda v: "" if pd.isna(v) else f"{v:.1f}")   # 고객이 없는 칸은 빈칸
    fig.add_trace(go.Heatmap(z=rate_grid.values, x=list(rate_grid.columns), y=list(rate_grid.index),
                             customdata=count_grid.values, colorscale="Blues", zmin=0, zmax=40,
                             text=cell_text.values, texttemplate="%{text}",
                             colorbar={"len": 0.3, "y": 0.72, "title": "%"},
                             hovertemplate="%{y} · %{x}<br>가입률 %{z}%<br>고객수 %{customdata}명<extra></extra>"),
                  row=2, col=1)
    fig.update_yaxes(autorange="reversed", row=2, col=1)

    # ③ 막대 4개
    positions = [(3, 1), (3, 4), (4, 1), (4, 4)]
    for (row, col), col_name in zip(positions, BAR_COLS):
        table = rate_table(data, col_name).reset_index()
        fig.add_trace(go.Bar(x=table[col_name].astype(str), y=table["rate_pct"], text=table["rate_pct"],
                             customdata=table["n_customers"], marker_color="#3776AB",
                             textposition="outside", textangle=0,
                             hovertemplate="%{x}<br>가입률 %{y}%<br>고객수 %{customdata}명<extra></extra>"),
                      row=row, col=col)
        bar = fig.data[-1]                          # 방금 그린 막대의 축에 전체 가입률 점선
        fig.add_shape(type="line", xref=f"{bar.xaxis} domain", x0=0, x1=1, yref=bar.yaxis,
                      y0=overall_rate, y1=overall_rate, line={"dash": "dot", "color": "gray"})
        fig.update_yaxes(range=[0, 100], row=row, col=col)

    fig.update_layout(title=f"[{job}] 고객 {n_customers:,}명 · 정기예금 가입률 {overall_rate:.1f}%",
                      height=1100, showlegend=False)
    return fig

print("데이터 준비 완료:", len(customers), "명")

## 1. 직업 고르기

선택 상자에서 직업을 고른 뒤 **아래 2번 셀**을 실행합니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

JOB_LIST = ["전체"] + sorted(customers["job_name"].unique().tolist())
job_picker = widgets.Dropdown(options=JOB_LIST, value="전체", description="직업")
display(job_picker)

## 2. 한 화면 대시보드 ⭐

위에서 고른 직업으로 그립니다. 직업을 바꾸면 이 셀을 다시 실행하세요.

In [ ]:
selected_job = job_picker.value              # 선택 상자에서 고른 직업
fig = full_dashboard(selected_job)
fig.show()

## 3. 고른 직업의 고객만 더 살펴보기

`selected_data`에 고른 직업의 고객만 담깁니다. 표를 바꿔 보려면 `# ←` 줄의 기준만 바꾸세요.

In [ ]:
selected_data = customers if selected_job == "전체" else customers[customers["job_name"] == selected_job]
print("선택한 직업:", selected_job, "/ 고객 수:", f"{len(selected_data):,}명")

view_col = "age_group"                      # ← "balance_group", "loan_status", "prev_campaign"으로 바꿔 보세요
rate_table(selected_data, view_col).query("n_customers > 0")   # 고객이 없는 줄은 빼고 보기

## 4. [선택] 대시보드를 파일로 저장

왼쪽 파일 창에 HTML 파일이 생깁니다. 내려받아 브라우저로 열면 마우스로 값을 볼 수 있습니다.

In [ ]:
file_name = f"dashboard_{selected_job}.html"
fig.write_html(file_name)
print("저장:", file_name)

> ⚠️ 이 데이터는 포르투갈 은행의 익명화된 공개 데이터(UCI Bank Marketing)입니다.
> 연령 · 직업 분석은 집단 단위 통계 교육용이며, 개인을 고르거나 배제하는 근거로 쓰지 않습니다.